In [2]:
import polars as pl
import os

ruta = "/Users/macbook/ProyectosLocales/PrecioLuz/datos"

archivos = [f for f in os.listdir(ruta) if f.startswith("gas_") and f.endswith(".xlsx")]

dfs = []
for a in archivos:
    dfs.append(pl.read_excel(f"{ruta}/{a}"))

df = pl.concat(dfs)

# fecha
df = df.with_columns(
    pl.col("Fecha").cast(pl.Date, strict=False)
).sort("Fecha")

print(df.head(5))
print(df.tail(5))
print("Shape:", df.shape)
print("Nulos:", df.null_count())

# duplicados
dup = df.group_by("Fecha").count().filter(pl.col("count") > 1)
print(dup)

# faltantes
f_min = df.select(pl.col("Fecha").min()).item()
f_max = df.select(pl.col("Fecha").max()).item()

fechas = pl.date_range(f_min, f_max, "1d", eager=True)
faltan = pl.DataFrame({"Fecha": fechas}).join(df, on="Fecha", how="anti")

print("Faltan:", faltan.height)

# limpiar columnas
df = df.rename({
    "Precio_gas\n[EUR/MWh]": "Precio_gas"
}).select(["Fecha", "Precio_gas"])

# guardar
df.write_csv("/Users/macbook/ProyectosLocales/PrecioLuz/datos/gas_total.csv")

print("csv guardado")

shape: (5, 2)
┌────────────┬────────────┐
│ Fecha      ┆ Precio_gas │
│ ---        ┆ [EUR/MWh]  │
│ date       ┆ ---        │
│            ┆ f64        │
╞════════════╪════════════╡
│ 2015-12-17 ┆ 19.67      │
│ 2015-12-18 ┆ 19.67      │
│ 2015-12-19 ┆ 19.4       │
│ 2015-12-20 ┆ 19.4       │
│ 2015-12-21 ┆ 18.55      │
└────────────┴────────────┘
shape: (5, 2)
┌────────────┬────────────┐
│ Fecha      ┆ Precio_gas │
│ ---        ┆ [EUR/MWh]  │
│ date       ┆ ---        │
│            ┆ f64        │
╞════════════╪════════════╡
│ 2025-12-27 ┆ 28.57      │
│ 2025-12-28 ┆ 28.75      │
│ 2025-12-29 ┆ 28.97      │
│ 2025-12-30 ┆ 28.23      │
│ 2025-12-31 ┆ 28.17      │
└────────────┴────────────┘
Shape: (3668, 2)
Nulos: shape: (1, 2)
┌───────┬────────────┐
│ Fecha ┆ Precio_gas │
│ ---   ┆ [EUR/MWh]  │
│ u32   ┆ ---        │
│       ┆ u32        │
╞═══════╪════════════╡
│ 0     ┆ 0          │
└───────┴────────────┘
shape: (0, 2)
┌───────┬───────┐
│ Fecha ┆ count │
│ ---   ┆ ---   │
│ date  ┆ 

/var/folders/hx/0zmhrp2d37x4jg59pl9ktf0r0000gn/T/ipykernel_53264/522483771.py:25: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  dup = df.group_by("Fecha").count().filter(pl.col("count") > 1)
